# Proyecto Final — Predicción de ingresos de películas con Spark ML

**Curso:** Big Data

**Dataset:** TMDB Movies (`data/movies.csv`, 1.485.329 filas, 24 columnas)

**Objetivo:** predecir `revenue` (ingresos en USD) de una película a partir de variables disponibles antes de su estreno, usando **Spark ML** como capa principal de Machine Learning.

## Índice

1. Definición del problema
2. Análisis de datos → problema de Machine Learning
3. Arquitectura e ingesta
4. Datos y análisis exploratorio (EDA)
5. Preparación e ingeniería de características
6. Baseline
7. Modelos
8. Evaluación y análisis de errores
9. Escalabilidad
10. Interpretación del modelo
11. Conclusiones
12. Reproducibilidad y declaración de IA

## Cómo ejecutar

```bash
uv sync                         # para sincronizar dependencias (útil luego de 'git pull')
uv run python src/data.py       # descarga data/movies.csv si falta
uv run jupyter lab              # abrir src/main.ipynb y ejecutar todo
```

## 1. Definición del problema

**Problema del dominio.** Estimar los ingresos (`revenue`) que generará una película a partir de información conocida antes de su estreno (presupuesto, duración, género, idioma, país/productora, año). Sirve para decisiones de inversión, compra de catálogo y priorización de producciones.

**Problema de Machine Learning.** Es una **regresión supervisada**: la variable objetivo es numérica y continua, y se busca minimizar el error de predicción sobre películas no vistas.

| Elemento | Definición |
| --- | --- |
| Unidad de análisis | Una película (una fila del dataset). |
| Variable objetivo | `revenue` en USD. |
| Predicción correcta | Error bajo y comparable contra un baseline; se usan RMSE, MAE y R² (en dólares y en escala logarítmica). |
| Aporte del modelo | Ordenar películas por potencial de ingresos y detectar casos atípicos. |
| Limitaciones | Solo ~0,7 % de las filas tienen `revenue` y `budget` válidos; los montos son autoinformados, no ajustados por inflación y no incluyen marketing. |

La variable `revenue` está muy sesgada (muchos ceros y valores extremos), por eso se modela `log1p(revenue)` y luego se vuelve a dólares para evaluar.


## 2. Análisis de datos → problema de Machine Learning

| Elemento previo | Transformación en el proyecto |
| --- | --- |
| Pregunta analítica: ¿qué explica la recaudación? | Variable objetivo: `revenue`; se busca una función de variables previas al estreno. |
| EDA y distribuciones | El EDA define el filtro de filas válidas y la transformación logarítmica del target. |
| Outliers y valores anómalos | Se filtran montos imposibles, ceros y filas con columnas corridas (ver sección 4). |
| Agregaciones | `n_genres` y `n_companies` resumen listas de géneros y productoras. |
| Particionamiento de datos | `local[4]` y `spark.sql.shuffle.partitions=8`; el experimento de escalabilidad varía el volumen de entrenamiento. |
| Streaming desde Kafka | No aplica: la arquitectura elegida es Spark directo sobre un CSV estático. |


## 3. Arquitectura e ingesta

**Vía principal elegida: A. Spark directo.** El dataset es un único CSV estático de ~655 MB. Spark puede leerlo, filtrarlo y procesarlo en paralelo sin necesidad de una capa intermedia. Dask o Kafka agregarían componentes sin un problema real que los justifique (el enunciado lo desaconseja explícitamente).

```
data/movies.csv
      |
      v
Spark DataFrame (local[4])
      |
      v
Limpieza + Feature Engineering
      |
      v
Spark ML Pipeline (StringIndexer / OneHotEncoder / VectorAssembler)
      |
      v
Predicción y evaluación (RMSE, MAE, R²)
```

No se utiliza Kafka, por lo que la sección 5.12 del enunciado no aplica. Todo el procesamiento y el modelo se ejecutan con `pyspark.ml` sobre DataFrames de Spark.


In [ ]:
import sys
import time
from pathlib import Path

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import (
    OneHotEncoder, StandardScaler, StandardScalerModel,
    StringIndexer, StringIndexerModel, VectorAssembler,
)
from pyspark.ml.regression import LinearRegression, RandomForestRegressor

# Configuración del entorno
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "data" / "movies.csv"

spark = (
    SparkSession.builder
    .master("local[4]")
    .appName("bigdata-revenue")
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Python:", sys.version.split()[0])
print("Spark :", spark.version)
print("Dataset:", DATA)
print(f"Tamaño : {DATA.stat().st_size / 1e6:,.0f} MB")
print("Cores  :", spark.sparkContext.defaultParallelism)

Picked up _JAVA_OPTIONS: -Djava.util.prefs.userRoot=/home/demian/.config/java


Picked up _JAVA_OPTIONS: -Djava.util.prefs.userRoot=/home/demian/.config/java


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/13 09:39:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Python: 3.12.16
Spark : 4.3.0
Dataset: /home/demian/dev/bigdata/data/movies.csv
Tamaño : 655 MB
Cores  : 4


## 4. Datos y análisis exploratorio (EDA)

Se lee el CSV completo como texto (`inferSchema=False`) para poder **detectar valores no numéricos** en lugar de que Spark los convierta silenciosamente en nulos. El dataset tiene 1.485.329 filas y 24 columnas.


In [2]:
raw = spark.read.csv(str(DATA), header=True, inferSchema=False)
raw.cache()

TOTAL = raw.count()
print(f"Filas: {TOTAL:,} | Columnas: {len(raw.columns)}")
raw.printSchema()

Filas: 1,485,329 | Columnas: 24
root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- vote_average: string (nullable = true)
 |-- vote_count: string (nullable = true)
 |-- status: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- adult: string (nullable = true)
 |-- backdrop_path: string (nullable = true)
 |-- budget: string (nullable = true)
 |-- homepage: string (nullable = true)
 |-- imdb_id: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- poster_path: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- spoken_languages: string (nullable = true

### 4.1 Nulos y vacíos por columna

Se cuenta como nulo tanto `null` como cadena vacía.

In [3]:
conteo_nulos = raw.select([
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
    for c in raw.columns
]).first().asDict()

nulos = spark.createDataFrame(
    [(c, conteo_nulos[c], round(100 * conteo_nulos[c] / TOTAL, 2)) for c in raw.columns],
    "columna string, nulos long, porcentaje double",
).orderBy(F.desc("nulos"), "columna")

nulos.show(len(raw.columns), truncate=False)

+--------------------+-------+----------+
|columna             |nulos  |porcentaje|
+--------------------+-------+----------+
|homepage            |1332970|89.74     |
|tagline             |1226863|82.6      |
|backdrop_path       |1121572|75.51     |
|keywords            |1103153|74.27     |
|production_companies|844633 |56.87     |
|imdb_id             |801367 |53.95     |
|production_countries|725739 |48.86     |
|spoken_languages    |700717 |47.18     |
|genres              |655478 |44.13     |
|poster_path         |520376 |35.03     |
|release_date        |347437 |23.39     |
|overview            |346413 |23.32     |
|original_language   |181    |0.01      |
|budget              |123    |0.01      |
|original_title      |57     |0.0       |
|revenue             |50     |0.0       |
|popularity          |26     |0.0       |
|title               |23     |0.0       |
|adult               |5      |0.0       |
|runtime             |1      |0.0       |
|vote_average        |1      |0.0 

**INTERPRETACIÓN:** Hay columnas con nulos masivos que no se van a usar: `homepage` (89,7 %), `tagline` (82,6 %), `backdrop_path` (75,5 %), `keywords` (74,3 %), `production_companies` (56,9 %), `imdb_id` (54,0 %), `production_countries` (48,9 %) y `spoken_languages` (47,2 %). `genres` falta en 44,1 % y `release_date` en 23,4 %, valores que sí afectan al modelado y se tratan en la limpieza.

Las columnas que finalmente usa el modelo tienen pocos nulos: `revenue` (50), `budget` (123), `runtime` (1) y `original_language` (181).

### 4.2 Duplicados y valores anómalos

In [4]:
print("Filas duplicadas exactas:", TOTAL - raw.dropDuplicates().count())
print("Filas con id duplicado  :", TOTAL - raw.select("id").distinct().count())
print("Ids distintos           :", raw.select("id").distinct().count())

print("\nValores de status:")
raw.groupBy("status").count().orderBy(F.desc("count")).show(10, truncate=False)

print("Valores no numéricos (no nulos) por columna:")
for c in ["revenue", "budget", "runtime"]:
    malos = raw.filter(
        F.col(c).isNotNull() & (F.trim(F.col(c)) != "")
        & F.expr(f"try_cast({c} as double)").isNull()
    ).count()
    print(f"  {c:10s} {malos:>6,}")

fechas_malas = raw.filter(
    F.col("release_date").isNotNull() & (F.trim(F.col("release_date")) != "")
    & ~F.col("release_date").rlike(r"^\d{4}-\d{2}-\d{2}$")
).count()
print(f"  release_date con formato inválido: {fechas_malas:,}")

print("\nEjemplos de filas con columnas corridas (status numérico):")
raw.filter(F.col("status").rlike(r"^\d+(\.\d+)?$")) \
   .select("id", "title", "status", "release_date", "revenue", "budget", "runtime") \
   .show(5, truncate=False)

Filas duplicadas exactas: 384


Filas con id duplicado  : 1346


Ids distintos           : 1483983

Valores de status:


+---------------+-------+
|status         |count  |
+---------------+-------+
|Released       |1428302|
|In Production  |25903  |
|Post Production|16994  |
|Planned        |12627  |
|Rumored        |901    |
|Canceled       |382    |
|0              |158    |
|0.0            |26     |
|1              |14     |
|3              |2      |
+---------------+-------+
only showing top 10 rows
Valores no numéricos (no nulos) por columna:


  revenue       153


  budget         83


  runtime        44


  release_date con formato inválido: 220

Ejemplos de filas con columnas corridas (status numérico):
+-------+--------------------------------------------------------+------+------------+----------+--------------------------------+-------+
|id     |title                                                   |status|release_date|revenue   |budget                          |runtime|
+-------+--------------------------------------------------------+------+------------+----------+--------------------------------+-------+
|345830 |"The Newbridge Tourism Board Presents: ""We're Newbridge|3     |Released    |2014-11-03|NULL                            |0      |
|593068 |"""Please Kill Me                                       |3     |Released    |1996-07-23|NULL                            |0      |
|699663 |"Hyper Projection Play ""Haikyuu!!"" Karasuno           |2     |Released    |2017-04-19|NULL                            |0      |
|1621702|"""Kya tu wahi soch raha hai                            

**INTERPRETACIÓN:** Hay 384 filas duplicadas exactas y 1.346 filas con `id` repetido, que se eliminan quedándose con una fila por `id`. También hay 153 valores no numéricos en `revenue`, 83 en `budget`, 44 en `runtime` y 220 fechas con formato inválido (`status`, `release_date`, `revenue`… corridos en algunas filas). Esos registros se descartan porque no permiten construir el target ni las features de forma confiable.

### 4.3 Variable objetivo: `revenue`

In [5]:
raw_num = raw.withColumn("revenue_num", F.expr("try_cast(revenue as double)"))
print("revenue nulo/vacío:", raw_num.filter(F.col("revenue").isNull() | (F.trim("revenue") == "")).count())
print("revenue = 0      :", raw_num.filter(F.col("revenue_num") == 0).count())
print("revenue > 0      :", raw_num.filter(F.col("revenue_num") > 0).count())

raw_num.filter(F.col("revenue_num") > 0).select(
    F.count("*").alias("n"),
    F.expr("round(min(revenue_num), 0)").alias("min"),
    F.expr("round(percentile_approx(revenue_num, 0.5), 0)").alias("mediana"),
    F.expr("round(percentile_approx(revenue_num, 0.99), 0)").alias("p99"),
    F.expr("round(max(revenue_num), 0)").alias("max"),
).show()

print("budget = 0:", raw.where(F.expr("try_cast(budget as double) = 0")).count())
print("budget > 0:", raw.where(F.expr("try_cast(budget as double) > 0")).count())

revenue nulo/vacío: 50


revenue = 0      : 1459888
revenue > 0      : 25237


+-----+---+--------+------------+-------------+
|    n|min| mediana|         p99|          max|
+-----+---+--------+------------+-------------+
|25237|1.0|626057.0|6.72806292E8|4.999999999E9|
+-----+---+--------+------------+-------------+

budget = 0: 1397986


budget > 0: 87137


**INTERPRETACIÓN:** El 98,3 % de las películas tiene `revenue = 0` (dato faltante codificado como cero): solo **25.237** filas (1,7 %) tienen ingresos positivos. Lo mismo ocurre con `budget`: 1.397.986 filas en cero y 87.137 positivas. Por eso el dataset de modelado se construye exigiendo `revenue > 0` y `budget > 0`; modelar los ceros como si fueran ingresos reales introduciría un sesgo enorme. También se descartan montos ≥ 4.000 millones (revenue) y ≥ 1.000 millones (budget) por ser centinelas o valores implausibles.

### 4.4 Construcción del dataset de modelado (embudo)

In [6]:
typed = (raw
    .withColumn("revenue", F.expr("try_cast(revenue as double)"))
    .withColumn("budget", F.expr("try_cast(budget as double)"))
    .withColumn("runtime", F.expr("try_cast(runtime as double)"))
    .withColumn("year", F.expr("try_cast(substring(release_date, 1, 4) as int)")))

filtros = []
acum = F.lit(True)
for nombre, cond in [
    ("Total", F.lit(True)),
    ("status = Released", F.col("status") == "Released"),
    ("revenue > 0 y < 4e9", (F.col("revenue") > 0) & (F.col("revenue") < 4e9)),
    ("budget > 0 y < 1e9", (F.col("budget") > 0) & (F.col("budget") < 1e9)),
    ("runtime 40-400", F.col("runtime").between(40, 400)),
    ("año 1900-2026", F.col("year").between(1900, 2026)),
]:
    acum = acum & cond if nombre != "Total" else cond
    filtros.append((nombre, acum))

embudo = typed.agg(*[
    F.sum(F.when(cond, 1).otherwise(0)).alias(nombre) for nombre, cond in filtros
]).first()

for nombre, _ in filtros:
    print(f"{nombre:22s} {embudo[nombre]:>10,}  {100 * embudo[nombre] / TOTAL:5.2f}%")

Total                   1,485,329  100.00%
status = Released       1,428,302  96.16%
revenue > 0 y < 4e9        24,839   1.67%
budget > 0 y < 1e9         17,840   1.20%
runtime 40-400             11,698   0.79%
año 1900-2026              10,774   0.73%


**INTERPRETACIÓN:** El embudo deja **10.774 películas (0,73 %)** con target, presupuesto, duración y fecha válidos. La deduplicación por `id` no elimina filas adicionales en este subconjunto, pero se mantiene como salvaguarda. Es una limitación central del dataset: la mayoría de las filas no tiene información financiera. Se documenta y se discute en las conclusiones; no se inventan valores para el target.

In [7]:
model_df = (typed
    .filter(F.col("status") == "Released")
    .filter((F.col("revenue") > 0) & (F.col("revenue") < 4e9))
    .filter((F.col("budget") > 0) & (F.col("budget") < 1e9))
    .filter(F.col("runtime").between(40, 400))
    .filter(F.col("year").between(1900, 2026))
    .dropDuplicates()
    .dropDuplicates(["id"]))

top_langs = [r[0] for r in model_df.groupBy("original_language").count()
             .orderBy(F.desc("count")).limit(10).collect()]

model_df = (model_df
    .withColumn("log_revenue", F.log1p("revenue"))
    .withColumn("log_budget", F.log1p("budget"))
    .withColumn("adult_flag", F.when(F.col("adult") == "True", 1).otherwise(0))
    .withColumn("n_genres", F.coalesce(F.size(F.split(F.col("genres"), ",")), F.lit(0)))
    .withColumn("n_companies", F.coalesce(F.size(F.split(F.col("production_companies"), ",")), F.lit(0)))
    .withColumn("main_genre", F.coalesce(F.trim(F.split(F.col("genres"), ",")[0]), F.lit("desconocido")))
    .withColumn("language", F.when(F.col("original_language").isin(top_langs),
                                   F.col("original_language")).otherwise("otro"))
    .select("id", "title", "year", "revenue", "log_revenue", "budget", "log_budget",
            "runtime", "adult_flag", "n_genres", "n_companies", "main_genre", "language"))
model_df.cache()

print(f"Filas de modelado: {model_df.count():,}")
model_df.select(
    F.round(F.min("revenue")).alias("revenue_min"),
    F.round(F.expr("percentile_approx(revenue, 0.5)")).alias("revenue_mediana"),
    F.round(F.max("revenue")).alias("revenue_max"),
    F.min("year").alias("año_min"),
    F.max("year").alias("año_max"),
).show()

model_df.select(
    F.round(F.min("budget")).alias("budget_min"),
    F.round(F.expr("percentile_approx(budget, 0.5)")).alias("budget_mediana"),
    F.round(F.max("budget")).alias("budget_max"),
    F.round(F.min("runtime")).alias("runtime_min"),
    F.round(F.expr("percentile_approx(runtime, 0.5)")).alias("runtime_mediana"),
    F.round(F.max("runtime")).alias("runtime_max"),
).show()

print("Correlación log(revenue) vs log(budget):",
      model_df.select(F.corr(F.log("revenue"), F.log("budget"))).first()[0])

print("\nIdiomas más frecuentes:", top_langs)
model_df.groupBy("main_genre").count().orderBy(F.desc("count")).show(12, truncate=False)

Filas de modelado: 10,774
+-----------+---------------+-----------+-------+-------+
|revenue_min|revenue_mediana|revenue_max|año_min|año_max|
+-----------+---------------+-----------+-------+-------+
|        1.0|    1.0920544E7|      3.0E9|   1913|   2026|
+-----------+---------------+-----------+-------+-------+



+----------+--------------+------------+-----------+---------------+-----------+
|budget_min|budget_mediana|  budget_max|runtime_min|runtime_mediana|runtime_max|
+----------+--------------+------------+-----------+---------------+-----------+
|       1.0|     8000000.0|8.74362803E8|       40.0|          104.0|      387.0|
+----------+--------------+------------+-----------+---------------+-----------+

Correlación log(revenue) vs log(budget): 0.8407924466285314

Idiomas más frecuentes: ['en', 'hi', 'fr', 'ru', 'es', 'ja', 'ta', 'zh', 'it', 'fa']


+---------------+-----+
|main_genre     |count|
+---------------+-----+
|Drama          |2406 |
|Comedy         |2128 |
|Action         |1525 |
|Horror         |668  |
|Adventure      |566  |
|Crime          |448  |
|Thriller       |444  |
|Romance        |312  |
|Animation      |292  |
|Science Fiction|236  |
|Documentary    |211  |
|Fantasy        |206  |
+---------------+-----+
only showing top 12 rows


**INTERPRETACIÓN:** Mediana de ingresos ≈ 11 millones de USD, presupuesto mediano ≈ 8 millones y duración mediana ≈ 104 minutos; hay valores extremos en ambos lados. La correlación entre `log(revenue)` y `log(budget)` es alta (≈ 0,84), lo que confirma que el presupuesto es la señal principal. Los géneros principales y los idiomas quedan listos para codificarse; las categorías poco frecuentes se agrupan en `otro`.

No se usan `vote_average`, `vote_count`, `popularity` ni `status` como features: son mediciones **posteriores al estreno** y producirían data leakage.

## 5. Preparación e ingeniería de características

**Target:** `log_revenue = log1p(revenue)` para reducir la asimetría; las métricas finales se informan en dólares.

**Features construidas (todas disponibles antes del estreno):**

| Feature | Construcción | Justificación |
| --- | --- | --- |
| `log_budget` | `log1p(budget)` | El presupuesto es la señal más fuerte y su efecto es multiplicativo. |
| `runtime` | duración en minutos | Proxy del tipo de producción. |
| `year` | año de estreno | Captura inflación y cambios del mercado. |
| `adult_flag` | `adult == "True"` | Segmento de contenido. |
| `n_genres` | cantidad de géneros | Películas con más géneros apuntan a audiencias más amplias. |
| `n_companies` | cantidad de productoras | Tamaño/estructura de la producción. |
| `main_genre` | primer género de la lista | Categoría principal del film. |
| `language` | idioma original (top 10 + `otro`) | Mercado principal. |

**Tratamiento de nulos y duplicados:** se eliminan filas sin target o sin presupuesto/fecha válidos, se deduplica por `id` y las categorías faltantes se asignan a `desconocido`/`otro`. No se imputa `revenue` (sería inventar el target).

### 5.1 Separación entrenamiento / evaluación

In [8]:
train = model_df.filter(F.col("year") <= 2015).cache()
test = model_df.filter(F.col("year") > 2015).cache()

print(f"Train: {train.count():,} películas (años {train.agg(F.min('year')).first()[0]}-2015)")
print(f"Test : {test.count():,} películas (años 2016-{test.agg(F.max('year')).first()[0]})")

Train: 8,329 películas (años 1913-2015)


Test : 2,445 películas (años 2016-2026)


Se usa una **separación temporal** (entrenar hasta 2015, evaluar desde 2016) porque el revenue depende del año por inflación y evolución del mercado; una división aleatoria mezclaría épocas y sobreestimaría la capacidad de generalización. El costo es que los modelos de árboles no extrapolan `year` fuera del rango de entrenamiento (lo discutimos en las conclusiones). No hay fuga de información: ninguna feature usa datos posteriores al estreno ni el target.

## 6. Baseline

Baseline de regresión: predecir siempre la **mediana** de `revenue` del conjunto de entrenamiento. Es una referencia razonable para datos sesgados y permite comprobar si los modelos aportan valor real.

In [9]:
evaluador_usd = RegressionEvaluator(labelCol="revenue", predictionCol="pred")
evaluador_log = RegressionEvaluator(labelCol="log_revenue", predictionCol="pred_log")

def medir(pred, nombre):
    return {
        "modelo": nombre,
        "rmse_usd": evaluador_usd.setMetricName("rmse").evaluate(pred),
        "mae_usd": evaluador_usd.setMetricName("mae").evaluate(pred),
        "r2_usd": evaluador_usd.setMetricName("r2").evaluate(pred),
        "rmse_log": evaluador_log.setMetricName("rmse").evaluate(pred),
        "r2_log": evaluador_log.setMetricName("r2").evaluate(pred),
    }

mediana = train.approxQuantile("revenue", [0.5], 0.001)[0]
base_pred = (test
    .withColumn("pred", F.lit(mediana))
    .withColumn("pred_log", F.log1p(F.lit(mediana))))

resultados = [medir(base_pred, "Baseline (mediana)")]
print(f"Mediana de revenue en train: ${mediana:,.0f}")

Mediana de revenue en train: $12,400,000


## 7. Modelos

Se comparan dos alternativas de Spark ML:

- **Linear Regression**: modelo lineal interpretable, con escalado de features (sin escalar, L-BFGS no converge bien por las magnitudes dispares).
- **Random Forest Regressor**: modelo no lineal que captura interacciones y no necesita escalado.

Ambos se entrenan dentro de un `Pipeline` que incluye indexado, one-hot encoding y ensamblado de features.

In [10]:
def crear_pipeline(modelo, escalar=False):
    etapas = [
        StringIndexer(inputCol="main_genre", outputCol="genre_idx", handleInvalid="keep"),
        StringIndexer(inputCol="language", outputCol="lang_idx", handleInvalid="keep"),
        OneHotEncoder(inputCols=["genre_idx", "lang_idx"],
                      outputCols=["genre_ohe", "lang_ohe"], dropLast=False),
        VectorAssembler(
            inputCols=["log_budget", "runtime", "year", "adult_flag",
                       "n_genres", "n_companies", "genre_ohe", "lang_ohe"],
            outputCol="features"),
    ]
    if escalar:
        etapas.append(StandardScaler(inputCol="features", outputCol="scaled",
                                     withMean=True, withStd=True))
        modelo.setFeaturesCol("scaled")
    etapas.append(modelo)
    return Pipeline(stages=etapas)

t0 = time.perf_counter()
lr_model = crear_pipeline(LinearRegression(labelCol="log_revenue"), escalar=True).fit(train)
lr_time = time.perf_counter() - t0

t0 = time.perf_counter()
rf_model = crear_pipeline(
    RandomForestRegressor(labelCol="log_revenue", numTrees=100, maxDepth=8, seed=42)
).fit(train)
rf_time = time.perf_counter() - t0

lr_pred = (lr_model.transform(test)
           .withColumn("pred", F.expm1("prediction"))
           .withColumn("pred_log", F.col("prediction")))
rf_pred = (rf_model.transform(test)
           .withColumn("pred", F.expm1("prediction"))
           .withColumn("pred_log", F.col("prediction")))

resultados.append({**medir(lr_pred, "Linear Regression"), "fit_s": round(lr_time, 2)})
resultados.append({**medir(rf_pred, "Random Forest"), "fit_s": round(rf_time, 2)})

tabla = spark.createDataFrame(resultados).select(
    "modelo",
    F.round("rmse_usd").cast("long").alias("rmse_usd"),
    F.round("mae_usd").cast("long").alias("mae_usd"),
    F.round("r2_usd", 4).alias("r2_usd"),
    F.round("rmse_log", 4).alias("rmse_log"),
    F.round("r2_log", 4).alias("r2_log"),
    "fit_s",
)
tabla.show(truncate=False)

+------------------+---------+--------+-------+--------+-------+-----+
|modelo            |rmse_usd |mae_usd |r2_usd |rmse_log|r2_log |fit_s|
+------------------+---------+--------+-------+--------+-------+-----+
|Baseline (mediana)|233684990|79715415|-0.0902|5.5707  |-0.2381|NULL |
|Linear Regression |198332150|61579662|0.2147 |2.3064  |0.7878 |2.89 |
|Random Forest     |203018970|61226314|0.1771 |2.4484  |0.7608 |15.78|
+------------------+---------+--------+-------+--------+-------+-----+



**INTERPRETACIÓN:** Los dos modelos superan ampliamente al baseline (R² negativo). En escala logarítmica, el R² sube a ≈ 0,79 (Linear Regression) y ≈ 0,76 (Random Forest), lo que indica que el modelo ordena bien las películas. En dólares el R² es mucho menor (≈ 0,21 / 0,18) porque unos pocos blockbusters concentran casi toda la suma de errores al cuadrado. El **mejor modelo global es Linear Regression**: menor RMSE y mayor R² en ambas escalas; Random Forest tiene un MAE levemente menor pero peor RMSE y R². Se elige Linear Regression por rendimiento y simplicidad.

## 8. Evaluación y análisis de errores

Además de las métricas, se revisan los residuales y los casos con mayor error para entender **dónde falla** el modelo.

In [11]:
mejor_pred = lr_pred

residuales = mejor_pred.withColumn("error", F.col("revenue") - F.col("pred"))
residuales.select(
    F.round(F.expr("percentile_approx(error, 0.05)"), 0).alias("error_p05"),
    F.round(F.expr("percentile_approx(error, 0.50)"), 0).alias("error_mediana"),
    F.round(F.expr("percentile_approx(error, 0.95)"), 0).alias("error_p95"),
).show()

print("Peores errores absolutos:")
(residuales
 .withColumn("error_abs", F.abs("error"))
 .select("title", "year", F.round("revenue").alias("revenue"),
         F.round("pred").alias("prediccion"), F.round("error_abs").alias("error_abs"))
 .orderBy(F.desc("error_abs"))
 .show(10, truncate=False))

+-----------+-------------+------------+
|  error_p05|error_mediana|   error_p95|
+-----------+-------------+------------+
|-1.744156E7|      53377.0|3.16468273E8|
+-----------+-------------+------------+

Peores errores absolutos:


+------------------------+----+-------------+------------+-------------+
|title                   |year|revenue      |prediccion  |error_abs    |
+------------------------+----+-------------+------------+-------------+
|TikTok Rizz Party       |2024|3.0E9        |1.83651173E8|2.816348827E9|
|Avengers: Endgame       |2019|2.8E9        |4.312072E8  |2.3687928E9  |
|Best Of Joy             |2024|2.0E9        |1692.0      |1.999998308E9|
|Avengers: Infinity War  |2018|2.052415039E9|2.89191108E8|1.763223931E9|
|Spider-Man: No Way Home |2021|1.921847111E9|1.86318115E8|1.735528996E9|
|IPL 2025                |2025|2.0E9        |2.97298342E8|1.702701658E9|
|Avatar: The Way of Water|2022|2.320250281E9|6.22618597E8|1.697631684E9|
|Codename 47             |2017|1.7E9        |1.0724867E8 |1.59275133E9 |
|The Lion King           |2019|1.663075401E9|2.02379663E8|1.460695738E9|
|Top Gun: Maverick       |2022|1.488732821E9|1.39518824E8|1.349213997E9|
+------------------------+----+-------------+------

**INTERPRETACIÓN:** La mediana del error es cercana a cero, pero la cola superior está dominada por películas con ingresos muy altos (Avengers, Avatar) y por registros sospechosos del dataset (por ejemplo, títulos con `revenue` de 2.000 o 3.000 millones y presupuesto mínimo). El modelo tiende a **subestimar los éxitos extremos**: con `log1p` aprende el comportamiento típico, no los outliers. Es un patrón esperable y se reporta como limitación, no se corrige eliminando casos del test.

## 9. Escalabilidad

Experimento: se entrena el mismo Random Forest con 25 %, 50 % y 100 % del conjunto de entrenamiento y se mide el tiempo de `fit` y el RMSE en el mismo test. Así se observa cómo cambia el costo al aumentar el volumen de datos.

In [12]:
escalabilidad = []
for fraccion in (0.25, 0.5, 1.0):
    sub = train if fraccion == 1.0 else train.sample(False, fraccion, seed=42)
    n_sub = sub.count()
    t0 = time.perf_counter()
    modelo = crear_pipeline(
        RandomForestRegressor(labelCol="log_revenue", numTrees=100, maxDepth=8, seed=42)
    ).fit(sub)
    tiempo = time.perf_counter() - t0
    pred = (modelo.transform(test)
            .withColumn("pred", F.expm1("prediction"))
            .withColumn("pred_log", F.col("prediction")))
    m = medir(pred, f"{int(fraccion * 100)} %")
    escalabilidad.append({
        "fraccion": f"{int(fraccion * 100)} %",
        "filas_train": n_sub,
        "fit_s": round(tiempo, 2),
        "rmse_usd": round(m["rmse_usd"]),
        "rmse_log": round(m["rmse_log"], 4),
    })

spark.createDataFrame(escalabilidad).show(truncate=False)
print("Particiones de shuffle:", spark.conf.get("spark.sql.shuffle.partitions"))

+-----------+-----+--------+--------+---------+
|filas_train|fit_s|fraccion|rmse_log|rmse_usd |
+-----------+-----+--------+--------+---------+
|2106       |3.94 |25 %    |2.541   |201656468|
|4201       |5.84 |50 %    |2.4783  |206068904|
|8329       |13.34|100 %   |2.4484  |203018970|
+-----------+-----+--------+--------+---------+

Particiones de shuffle: 8


**INTERPRETACIÓN:** El tiempo de entrenamiento crece de forma aproximadamente lineal con el volumen (≈ 4 s → 6 s → 13 s para 2,1k → 4,2k → 8,3k filas), mientras que el RMSE se mantiene estable. El cuello de botella está en el **entrenamiento del Random Forest** y en los *shuffles* del one-hot encoding, no en la lectura del CSV ni en la evaluación. Con `local[4]` el paralelismo está limitado a 4 hilos; en un clúster el costo se repartiría entre ejecutores. El experimento se hizo sobre el subconjunto modelable; la lectura y el EDA sí procesan las 1,48 M de filas completas.

## 10. Interpretación del modelo

Se analizan las importancias de Random Forest (contraste no lineal) y los coeficientes de Linear Regression (modelo elegido), para explicar **qué variables resultan relevantes**.

In [13]:

indexadores = {s.getOutputCol(): s for s in rf_model.stages if isinstance(s, StringIndexerModel)}

nombres = ["log_budget", "runtime", "year", "adult_flag", "n_genres", "n_companies"]
nombres += [f"main_genre={v}" for v in indexadores["genre_idx"].labels] + ["main_genre=no_visto"]
nombres += [f"language={v}" for v in indexadores["lang_idx"].labels] + ["language=no_visto"]

importancias = rf_model.stages[-1].featureImportances.toArray()
assert len(nombres) == len(importancias), (len(nombres), len(importancias))

top = sorted(zip(nombres, importancias), key=lambda x: -x[1])[:12]
spark.createDataFrame(
    [(n, round(float(v), 4)) for n, v in top],
    "variable string, importancia double",
).show(truncate=False)

escalador = [s for s in lr_model.stages if isinstance(s, StandardScalerModel)][0]
coef_escalados = lr_model.stages[-1].coefficients.toArray()
desvios = escalador.std.toArray()
coeficientes = [c / s if s else 0.0 for c, s in zip(coef_escalados, desvios)]
spark.createDataFrame(
    [(n, round(float(c), 4)) for n, c in zip(nombres[:6], coeficientes[:6])],
    "variable string, coeficiente double",
).show(truncate=False)

+----------------------+-----------+
|variable              |importancia|
+----------------------+-----------+
|log_budget            |0.7066     |
|n_companies           |0.0922     |
|runtime               |0.0497     |
|year                  |0.035      |
|language=en           |0.0325     |
|language=otro         |0.016      |
|n_genres              |0.0152     |
|main_genre=Drama      |0.0051     |
|main_genre=desconocido|0.0047     |
|main_genre=Documentary|0.0033     |
|main_genre=Comedy     |0.003      |
|adult_flag            |0.0028     |
+----------------------+-----------+

+-----------+-----------+
|variable   |coeficiente|
+-----------+-----------+
|log_budget |0.8931     |
|runtime    |0.0083     |
|year       |-0.0174    |
|adult_flag |0.6848     |
|n_genres   |-0.0135    |
|n_companies|0.044      |
+-----------+-----------+



**INTERPRETACIÓN:** `log_budget` concentra ≈ 71 % de la importancia del Random Forest; le siguen `n_companies` (≈ 9 %), `runtime` (≈ 5 %) y `year` (≈ 3,5 %). Los géneros y el idioma aportan poco, aunque `language=en` sí aparece. En la regresión, el coeficiente de `log_budget` es ≈ +0,89 sobre `log(revenue)`: un aumento de 1 % en el presupuesto se asocia con ≈ 0,89 % más de ingresos, manteniendo el resto constante. Es una asociación, no una relación causal.

## 11. Conclusiones

**Respuestas a las preguntas mínimas del enunciado:**

1. **¿Qué problema real intenta resolver?** Estimar los ingresos de una película antes del estreno para apoyar decisiones de inversión y catálogo.
2. **¿Por qué es un problema de Machine Learning?** La relación entre presupuesto, duración, género, idioma y año con el revenue es compleja, no lineal y ruidosa; un modelo supervisado mejora una regla fija.
3. **¿Cuál es la variable objetivo?** `revenue` en USD (modelado como `log1p(revenue)`).
4. **¿Qué representa cada observación?** Una película del dataset TMDB.
5. **¿Qué variables utiliza el modelo?** `log_budget`, `runtime`, `year`, `adult_flag`, `n_genres`, `n_companies`, `main_genre` y `language`.
6. **¿Cómo se construyeron?** Logaritmo del presupuesto, conteos de listas (géneros/productoras), primer género, top-10 de idiomas + `otro`, y codificación one-hot en el pipeline.
7. **¿Cuál es el baseline?** Predecir la mediana de revenue del train (12,4 M USD): RMSE ≈ 233,7 M, MAE ≈ 79,7 M, R² ≈ −0,09.
8. **¿Qué modelos se compararon?** Linear Regression y Random Forest (con pipeline de Spark ML).
9. **¿Qué métrica es la más importante?** RMSE en dólares, porque penaliza los errores grandes que importan en una decisión de inversión; MAE y R² se reportan como complemento y el RMSE logarítmico para comparar la tendencia central.
10. **¿Existe desbalance, leakage o dependencia temporal?** Sí: el 98,3 % de los revenue son ceros (se excluyen), se evitó leakage excluyendo variables post-estreno y se usó separación temporal (≤2015 / >2015). Limitación: los árboles no extrapolan el año.
11. **¿Cuál fue el principal cuello de botella?** El entrenamiento del Random Forest y los shuffles del one-hot encoding; la lectura del CSV y el EDA son costosos en tiempo total pero no dominan el entrenamiento.
12. **¿Cómo cambia el rendimiento al aumentar la escala?** El tiempo de `fit` crece casi linealmente (≈ 4 s, 6 s y 13 s para 25 %, 50 % y 100 % del train) y el RMSE se mantiene estable (≈ 200 M).
13. **¿Qué limitaciones tiene la solución?** Solo el 0,73 % de las filas es modelable; montos autoinformados y no ajustados por inflación; outliers y registros sospechosos; no incluye marketing, elenco ni crítica; el modelo subestima blockbusters.
14. **¿Qué decisión podría tomarse con el resultado?** Priorizar producciones por potencial de ingresos, filtrar proyectos de alto riesgo y usar el modelo como *screening*, nunca como predicción exacta.

**Cierre.** El pipeline completo (lectura → limpieza → features → Spark ML → evaluación → escalabilidad) supera el baseline y es reproducible. La principal deuda es la calidad de los datos financieros, no el algoritmo.


## 12. Reproducibilidad y declaración de IA

**Reproducibilidad**

- Python 3.12.16 y Spark 4.3.0, dependencias fijadas en `pyproject.toml` y `uv.lock`.
- Semilla 42 en splits, muestreos y modelos; separación temporal fija en 2015.
- `spark.sql.shuffle.partitions = 8` y `local[4]` para que el experimento sea repetible en esta máquina.
- Comandos:
  ```bash
  uv sync
  uv run python src/data.py
  uv run jupyter lab
  ```

**Declaración de uso de IA.** Se utilizó un asistente de IA (OpenCode) para estructurar el notebook y redactar las secciones explicativas. Todo el código fue ejecutado de principio a fin sobre el dataset real (`nbconvert --execute`) y las métricas reportadas provienen de esa ejecución; la interpretación de resultados se verificó contra las tablas generadas.

**Entregables relacionados:** `docs/requerimientos.tex` (enunciado), `src/data.py` (descarga del dataset), `src/main.ipynb` (este notebook) y `README.md`.
